# Computer lab 1: linear and integer optimization

**Mathematical Optimization, VU Amsterdam, 2026–27.**

Formulate each problem before implementing it in Pyomo. State what each variable
measures and whether it is continuous, integer or binary. Interpret the solver's
result in the units of the problem.

## Setup

Run the next two cells before starting. They prepare HiGHS and the course
helpers. On Colab, expand the setup form only if you want to inspect its code.
If asked to restart the session, restart and run Setup again. See the
[course setup instructions](https://github.com/alessandrozocca/MO2026#computer-lab-setup)
for local use and troubleshooting.

In [ ]:
#@title Prepare course helpers and solvers
import hashlib
import os
from pathlib import Path
import sys
import types
from urllib.request import urlopen

SETUP_COMMIT = '87cc1970669338f2009a45f97051d5a71d5096f2'
SETUP_SHA256 = "257b7edfe96e804b27ceb6b50c6c0772fbe21749806f2b7e5e17b32d3dd631c9"
local_repo = os.environ.get("MO2026_REPO")
if local_repo:
    setup_location = str(Path(local_repo).expanduser() / "lab_setup.py")
    setup_source = Path(setup_location).read_bytes()
else:
    setup_location = f"https://raw.githubusercontent.com/alessandrozocca/MO2026/{SETUP_COMMIT}/lab_setup.py"
    with urlopen(setup_location, timeout=30) as response:
        setup_source = response.read()
    if hashlib.sha256(setup_source).hexdigest() != SETUP_SHA256:
        raise RuntimeError("Course setup checksum mismatch; the downloaded code was not executed.")

lab_setup = types.ModuleType("mo2026_lab_setup")
lab_setup.__file__ = setup_location
sys.modules[lab_setup.__name__] = lab_setup
exec(compile(setup_source, setup_location, "exec"), lab_setup.__dict__)
if local_repo and lab_setup.API_VERSION != 1:
    raise RuntimeError("This notebook requires course setup API 1.")
lab = lab_setup.setup(week=1, check=False, verbose=False)
solve = lab_setup.solve
relax_integrality = lab_setup.relax_integrality
format_number = lambda value, digits=3: lab_setup.format_number(value, digits)
solver = lab.lp_solver
lab_helpers = lab.helpers['tutorial1']
draw_graph = lab_helpers.draw_graph
draw_network = lab_helpers.draw_network
display_side_by_side = lab_helpers.display_side_by_side

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyomo.environ as pyo
from IPython.display import display

## Reading

Before starting, read [Chapter 1: Mathematical Optimization](https://mobook.github.io/MO-book/notebooks/01/01.00.html)
of *Hands-On Mathematical Optimization with Python*. Useful examples also appear in
[Chapter 2: Linear Optimization](https://mobook.github.io/MO-book/notebooks/02/02.00.html),
[Chapter 3: Mixed Integer Linear Optimization](https://mobook.github.io/MO-book/notebooks/03/03.00.html)
and [Chapter 4: Network Optimization](https://mobook.github.io/MO-book/notebooks/04/04.00.html).

The [Pyomo expression documentation](https://pyomo.readthedocs.io/en/6.10.1/explanation/modeling/math_programming/expressions.html)
explains how Python expressions become model expressions.

## Exercise 1: a first linear program

Implement and solve the following LP with Pyomo. Report the optimal decisions and
objective value.

$$
\begin{aligned}
\max\quad &3x_1+4x_2\\
\text{s.t.}\quad &x_1+2x_2\le 6,\\
&x_2\le 2,\\
&2x_1+x_2\le 9,\\
&x_1,x_2\ge 0.
\end{aligned}
$$

Check termination before loading a solution, then evaluate it with `pyo.value`.
The function below is for models where an optimum is expected. Exercise 6(a)
handles infeasibility and limited solves separately. Number formatting changes
only displayed text.

In [ ]:
def solve_and_load(model, solver=solver):
    result = solver.solve(model, load_solutions=False)
    pyo.assert_optimal_termination(result)
    model.solutions.load_from(result)
    return result

In [ ]:
# Uncomment this skeleton and replace each ... with the corresponding expression.
# lp_model = pyo.ConcreteModel(name="Exercise 1")
# lp_model.x = pyo.Var([1, 2], domain=pyo.NonNegativeReals)
# lp_model.objective = pyo.Objective(expr=..., sense=pyo.maximize)
# lp_model.first = pyo.Constraint(expr=... <= 6)
# lp_model.second = pyo.Constraint(expr=... <= 2)
# lp_model.third = pyo.Constraint(expr=... <= 9)
#
# result = solve_and_load(lp_model)
# print(f"Termination: {result.solver.termination_condition}")
# print(f"x1 = {pyo.value(lp_model.x[1]):.3f}; x2 = {pyo.value(lp_model.x[2]):.3f}")
# print(f"Optimal value: {pyo.value(lp_model.objective):.3f}")

## Exercise 2: blending sausages

A producer must supply 350 economy sausages and 500 premium sausages. Each sausage
weighs 50 g (0.05 kg). Economy sausages must contain **at least 40% pork by mass**;
premium sausages must contain **at least 60% pork by mass**. Each type may contain
at most 25% starch by mass. These are requirements of this exercise.

| Ingredient | Cost (€/kg) | Total availability (kg) |
|:---|---:|---:|
| Pork | 4.32 | 30 |
| Wheat | 2.46 | 20 |
| Starch | 1.86 | 17 |

The producer has already paid for 23 kg of pork and must use all of it. The 30 kg
availability **includes** these 23 kg; at most 7 kg more pork can be purchased at
the listed price. Wheat and starch must still be purchased. There are no other
costs or losses.

Formulate an LP that chooses the mass of each ingredient in each sausage type
and minimizes **additional purchasing cost**. Solve it and report a recipe in kg,
the additional cost, and the total ingredient cost including the prepaid pork.

In [ ]:
ingredient_cost = {"pork": 4.32, "wheat": 2.46, "starch": 1.86}
availability = {"pork": 30, "wheat": 20, "starch": 17}
sausage_count = {"economy": 350, "premium": 500}
required_kg = {kind: 0.05 * count for kind, count in sausage_count.items()}
minimum_pork_share = {"economy": 0.40, "premium": 0.60}
maximum_starch_share = 0.25
prepaid_pork_kg = 23

In [ ]:
# Use one nonnegative variable for each (ingredient, sausage type) pair.
# Add mass balances, composition requirements, stock limits and the pork contract.
# Minimize total ingredient cost minus the prepaid pork cost.
# Solve and report a recipe and both cost measures.

## Exercise 3: minimizing the latest completion time

A company runs two projects simultaneously. Without extra funding, project 1 takes
200 days and project 2 takes 100 days. Funding is allocated in **whole blocks of
€10,000**. A block allocated to project 1 shortens it by 3 days; one allocated
to project 2 shortens it by 2 days. The total budget is €500,000. The company
wants both projects finished as early as possible.

With $x_i$ the number of funding blocks allocated to project $i$, a formulation is

$$
\begin{aligned}
\min\quad &\max\{200-3x_1,\;100-2x_2\}\\
\text{s.t.}\quad &x_1+x_2\le50,\\
&x_1,x_2\in\mathbb{Z}_{\ge0}.
\end{aligned}
$$

1. Try using Python's `max()` on the two Pyomo expressions. Explain the error.
2. Introduce a variable for the latest completion time and obtain a
   mixed-integer linear formulation. Implement it and report the spending on each
   project and the resulting completion times.

*Hint: use the epigraph reformulation from Tutorial sheet 1, Exercise 1.*

In [ ]:
# Create two Pyomo variables and try max(200 - 3*x[1], 100 - 2*x[2]).
# Import PyomoException with: from pyomo.common.errors import PyomoException
# Catch the expected PyomoException so Run All can continue.
# Explain why Python cannot compare these symbolic expressions.

In [ ]:
# Introduce the latest completion time T and two epigraph inequalities.
# Keep whole funding blocks as nonnegative integers and enforce the budget.
# Solve and convert funding blocks back to euros.

## Exercise 4: minimum-cost flow

For a directed network with nodes $V$ and arcs $E$, solve

$$
\begin{aligned}
\min\quad &\sum_{(i,j)\in E}c_{ij}x_{ij}\\
\text{s.t.}\quad
&\sum_{j:(v,j)\in E}x_{vj}-\sum_{i:(i,v)\in E}x_{iv}=b_v
&&\text{for each }v\in V,\\
&0\le x_{ij}\le u_{ij} &&\text{for each }(i,j)\in E.
\end{aligned}
$$

Here $x_{ij}$ is a continuous flow, $u_{ij}$ is its capacity, and $c_{ij}$ is its
cost per unit. Positive $b_v$ means supply and negative $b_v$ means demand:
**outgoing minus incoming flow equals $b_v$**.

Implement a function `build_mcf(network)` that returns an unsolved Pyomo model.
Use it for both instances below. Report the cost and an arc-flow table for each,
and draw the flows with `draw_network(network, ax=..., edge_flows=...)`.
Each ordered pair of nodes identifies a single arc in these data.

In [ ]:
network_1 = {
    "nodes": {1: {"b": 1}, 2: {"b": 2}, 3: {"b": 1}, 4: {"b": -4}},
    "edges": {
        (1, 2): {"u": 4, "c": 1},
        (2, 3): {"u": 5, "c": 4},
        (3, 4): {"u": 5, "c": 4},
        (1, 4): {"u": 2, "c": 1},
        (2, 4): {"u": 2, "c": 0},
    },
}

network_2 = {
    "nodes": {
        0: {"b": 15}, 1: {"b": 0}, 2: {"b": 0}, 3: {"b": 0},
        4: {"b": -1}, 5: {"b": -1}, 6: {"b": -2}, 7: {"b": -2},
        8: {"b": -2}, 9: {"b": -2}, 10: {"b": -5},
    },
    "edges": {
        (0, 1): {"u": 15, "c": 1},
        (1, 2): {"u": 15, "c": 1},
        (2, 9): {"u": 10, "c": 1},
        (2, 3): {"u": 15, "c": 1},
        (3, 4): {"u": 10, "c": 3},
        (3, 5): {"u": 10, "c": 2},
        (4, 7): {"u": 4, "c": 1},
        (4, 8): {"u": 5, "c": 4},
        (5, 8): {"u": 5, "c": 3},
        (5, 9): {"u": 6, "c": 2},
        (5, 10): {"u": 5, "c": 1},
        (8, 6): {"u": 8, "c": 3},
        (6, 10): {"u": 1, "c": 3},
        (7, 8): {"u": 4, "c": 2},
        (7, 10): {"u": 2, "c": 2},
        (8, 10): {"u": 5, "c": 3},
        (9, 10): {"u": 3, "c": 1},
    },
}

for name, network in [("Network 1", network_1), ("Network 2", network_2)]:
    print(f"{name}: {len(network['nodes'])} nodes, {len(network['edges'])} arcs; "
          f"net supply = {sum(data['b'] for data in network['nodes'].values())}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6), layout="constrained")
for ax, network, title in zip(axes, [network_1, network_2], ["Network 1", "Network 2"]):
    draw_network(network, ax=ax)
    ax.set_title(title + ": capacities u and unit costs c")
plt.show()

In [ ]:
# def build_mcf(network):
#     model = pyo.ConcreteModel(name="Minimum-cost flow")
#     # Index continuous nonnegative flows by the arcs of this network.
#     # Add the cost objective, node balances and arc capacity limits.
#     return model

In [ ]:
# Build and solve each instance separately with solve_and_load(model).
# Extract full-precision flows with pyo.value(...), report costs and draw the flows.

## Exercise 5: graph coloring

Find a label **A**, **B** or **C** for every vertex of each graph below so that
adjacent vertices have different labels. Graph 1 is the octahedral graph
$K_{2,2,2}$; Graph 2 is the Petersen graph.

Formulate a binary integer model, implement `build_coloring(graph)`, and display
one feasible labeling of each graph. Every vertex must have **exactly one**
label. This is a feasibility problem: a constant objective of zero is sufficient.
You are not asked to minimize the number of colors.

In [ ]:
graph_1 = {
    "nodes": list(range(6)),
    "edges": [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 0),
              (0, 2), (2, 4), (4, 0), (1, 3), (3, 5), (5, 1)],
}
graph_2 = {
    "nodes": list(range(10)),
    "edges": [(0, 1), (1, 2), (2, 3), (3, 4), (4, 0),
              (0, 5), (1, 6), (2, 7), (3, 8), (4, 9),
              (5, 7), (7, 9), (9, 6), (6, 8), (8, 5)],
}

fig, axes = plt.subplots(1, 2, figsize=(11, 5), layout="constrained")
for ax, graph, title in zip(axes, [graph_1, graph_2], ["Octahedral graph", "Petersen graph"]):
    draw_graph(graph, ax=ax)
    ax.set_title(title)
plt.show()

In [ ]:
# Create binary variables indexed by (vertex, label).
# Enforce exactly one label per vertex and different labels across each edge.
# Solve with objective 0. Extract binary decisions using a tolerance (> 0.5).
# Color the vertices by label with draw_graph(graph, ax=..., node_labels=labels).

## Exercise 6: container shipping

A company has ships $i\in I$, with capacities $c_i$ containers, and must collect
$d_j$ containers from each port $j\in J$. Several ships may collect from the same
port. Ship $i$ incurs a fixed cost $f_{ij}$ whenever it visits port $j$, regardless
of the number collected there.

Treat each ship's capacity as a limit on its **total** pickup across all ports.
Routes, sailing order and unloading during the collection period are outside
this model. Containers are indivisible.

### 6(a): choosing visits and pickups

Formulate and solve an integer linear program that minimizes total visit cost
using the data below. Infer the ships and ports from the data labels.

1. Introduce visit binaries and pickup quantities. Derive a valid upper bound
   $M_{ij}$ for the linking constraint from the capacities and demands. What are
   its units? Also find the smallest uniform bound justified by these individual
   capacity and demand limits.
2. Solve with uniform $M=10,50,100,130,500,1000,10000$, and with your bounds
   $M_{ij}$. Identify which choices preserve the full intended feasible set.
   Handle infeasibility before trying to read a solution.
3. For each formulation, report its variable/constraint counts, plain LP value,
   processed root bound, LP/root gaps relative to the full MILP optimum,
   full-solve node count and observed time. Use a fresh model for every run and
   relax **all** integer variables for the LP. Limit the root run with
   `options={"mip_max_nodes": 1}`. Distinguish a proven bound from an incumbent,
   which may be absent. Explain why a stronger LP bound need not mean a faster
   solve.
4. Display an optimal integer shipping plan and, separately, the fractional LP
   solution for $M=10000$. Keep fractional visit values visible.
5. Add $\sum_i x_{ij}\ge1$ for every positive-demand port. Prove validity for
   the integer model. Before solving, predict which formulations these rows can
   strengthen. Compare $M=10000$ and pair-specific linking with and without them.
6. Identify the most violated visit row in the $M=10000$ LP and explain why its
   identity does not depend on which optimal LP solution is returned. Prove that
   the pair-specific LP already implies every visit row.
7. Interpret the supplied root-log excerpt. Why can its first LP value differ
   from the plain LP, and why can a reported integer bound differ from the last
   fractional bound in the log?

Use HiGHS with its default presolve and cuts for all these comparisons. A single
timing is an observation about this run, not a reliable ranking of formulations.
On this small instance, presolve and cuts can largely hide the runtime effect of
the different bounds; the LP relaxation values show their strength more clearly.

In [ ]:
capacity_values = [100, 150, 200, 250, 300]
demand_values = [48, 53, 77, 74, 98, 103, 108, 114, 122, 130]
capacity = pd.Series(capacity_values,
                     index=[f"Ship {i + 1}" for i in range(len(capacity_values))],
                     name="Capacity (containers)")
demand = pd.Series(demand_values,
                   index=[f"Port {j + 1}" for j in range(len(demand_values))],
                   name="Demand (containers)")
cost = pd.DataFrame(
    [[115, 368, 112, 466, 206],
     [397, 240, 303, 411, 371],
     [195, 228, 433, 258, 432],
     [478, 185, 306, 422, 475],
     [342, 417, 438, 281, 355],
     [202, 342, 228, 375, 379],
     [180, 113, 331, 355, 116],
     [239, 208, 381, 484, 323],
     [344, 301, 249, 224, 419],
     [258, 419, 347, 293, 254]],
    columns=capacity.index, index=demand.index,
)
display_side_by_side(
    [cost, capacity.to_frame(), demand.to_frame()],
    ["Visit costs f (ports × ships)", "Ship capacities c", "Port demands d"],
)
print(f"Total demand: {demand.sum()} containers; total capacity: {capacity.sum()} containers")

The cost table has ports as rows and ships as columns: the mathematical
$f_{ij}$ (ship $i$, port $j$) is accessed as `cost.loc[j, i]` in Python.

In [ ]:
# def build_shipping(capacity, demand, cost, upper):
#     # Use data labels as indices; upper is a dictionary keyed by (ship, port).
#     # Add visit binaries, integer pickups, cost, linking, capacity and demand rows.
#     return model

`solve` returns `status`, `objective`, `bound`, `gap`, `nodes` and `time`.
An `optimal` status certifies the optimum. A limited run can load an incumbent
without proving it optimal; its status can be `other`, and `objective` can be
`None`. Inspect the status before reading model values. For minimization, the
root bound is a lower bound; it need not equal the plain LP value.

Use the reporting scaffold below for shipping and part 6(c). Its LP/root gaps
are percentages relative to the **proven full MILP optimum**, when available.
The helper's own `gap` is instead a fraction relative to that run's incumbent.

In [ ]:
def compare_formulation(build):
    models, reports = {}, {}
    for kind in ("LP", "Root", "MILP"):
        model = build()
        if kind == "LP":
            relax_integrality(model)
        options = {"mip_max_nodes": 1} if kind == "Root" else None
        reports[kind] = solve(model, options=options)
        models[kind] = model
    full = reports["MILP"]
    optimum = full["objective"] if full["status"] == "optimal" else None
    def gap_percent(bound):
        if optimum is None or bound is None:
            return None
        return 100 * (optimum - bound) / max(abs(optimum), 1e-12)
    row = {
        "Variables": sum(1 for _ in models["MILP"].component_data_objects(pyo.Var)),
        "Rows": sum(1 for _ in models["MILP"].component_data_objects(pyo.Constraint, active=True)),
        "LP status": reports["LP"]["status"], "LP value": reports["LP"]["objective"],
        "LP gap (%)": gap_percent(reports["LP"]["objective"]),
        "Root status": reports["Root"]["status"], "Root bound": reports["Root"]["bound"],
        "Root incumbent": reports["Root"]["objective"],
        "Root gap (%)": gap_percent(reports["Root"]["bound"]),
        "MILP status": full["status"], "MILP value": full["objective"],
        "MILP nodes": full["nodes"], "MILP seconds": full["time"],
    }
    return row, models, reports

In [ ]:
# Construct the uniform and pair-specific upper bounds.
# Call compare_formulation with a builder for each choice; keep every model fresh.
# Label restricted/infeasible choices separately from equivalent valid formulations.
# Report the fixed columns in the scaffold, including status before values.

In [ ]:
# Display the optimal integer plan obtained using the pair-specific bounds.
# Retain raw pickup values and format a display copy as whole containers.

In [ ]:
# Display the LP solution for M = 10000 using decimal visit values.
# Explain the difference between its lower bound and an implementable plan.

### Visit inequalities and root processing

Implement `add_visit_rows(model)` for positive-demand ports. Predict the effect
on the two LP relaxations before running the comparison. Which visit row is most
violated in the $M=10000$ LP? Justify the answer using the positive visit costs.

In [ ]:
# Add one visit row per positive-demand port and compare both formulations.
# Sum the fractional visits by port; identify and explain the largest violation.
# Prove validity for the integer model and implication by the pair-specific LP.

The following abbreviated **reference root log** uses HiGHS 1.15.1 with
$M=10000$ and default presolve/cuts. Read it alongside your measured table.

| Logged event | Interpretation |
|:---|:---|
| Presolve: `Not reduced` | No row/column reduction is reported; this does not rule out coefficient strengthening. |
| First root LP: 2125.68; zero cuts | Processing has already strengthened the plain relaxation. |
| Subsequent cut rows raise the bound | These are processed bounds, not new plain-LP values. |
| `L` finds incumbent 2449 | A sub-MIP heuristic found a feasible integer solution. |
| Restart after 4% inactive integer columns | HiGHS changes the remaining search representation. |
| Last fractional bound about 2203.30; reported bound 2204 | The objective is integer-valued, so its lower bound can be rounded upward. |

The plain $M=10000$ LP has value 20. Equal processed bounds across formulations
do not identify which internal strengthening operation was responsible.
A root-limited status of `other` is compatible with a valid incumbent and bound;
it does not prove optimality. Optional: turn presolve off for one diagnostic
comparison rather than repeating the entire sweep.

### 6(b): minimizing the fleet size

For a long-term decision, the company may obtain any number of ships of any of the
five capacity types in part (a). It wants to minimize the **number of ships**;
purchase prices and port visit costs are ignored. Now each port's entire demand
must be collected by **exactly one ship**. A ship may serve several ports, subject
to its capacity.

Explain why the model can restrict attention to the largest capacity type. Then
formulate and solve an integer model with individual candidate ships and port
assignment variables. Give a lower bound on the number of ships and display a
feasible allocation attaining the optimum.

In [ ]:
# Explain the dominance of the largest type under the stated objective.
# Index candidate ships individually, with binaries for ship use and port assignments.
# Enforce one ship per port and a capacity limit for each used ship.
# Solve, list each used ship's ports and load, and compare with a capacity bound.

### 6(c): aggregated versus individual linking

An uncapacitated facility-location model opens facilities $j\in J$ at costs
$F_j$ and allocates each customer $k\in K$ among open facilities. Let
$o_j\in\{0,1\}$ indicate an opening and let $s_{jk}\in[0,1]$ be the allocation
fraction. Minimize

$$
\sum_j F_jo_j+\sum_{j,k}C_{jk}s_{jk},\qquad \sum_j s_{jk}=1\quad(k\in K).
$$

Compare aggregated linking $\sum_k s_{jk}\le |K|o_j$ with individual linking
$s_{jk}\le o_j$ for every $j,k$.

1. Complete `build_location(data, strong=False)` using the supplied scaffold.
   Explain why the two formulations are equivalent for binary openings, why the
   individual-linking LP is contained in the aggregated-linking LP, and why the
   reverse inclusion fails. Give a fractional counterexample.
2. Predict and compare LP and MILP values for the two-facility example. Then use
   the fixed larger instance, preserving the order of its random-generator calls.
   Reuse the reporting scaffold from 6(a), including processed root bounds,
   model sizes and full-solve effort. Do not infer a runtime ranking from one run.
3. Explain why continuous allocation fractions lose nothing for fixed openings.
   Does an integral optimum of the strong LP on these data prove that its
   polytope is integral in general?

Lecture 2 uses $y_j,x_{ij},c_j,h_{ij}$ for our $o_j,s_{ji},F_j,C_{ji}$,
respectively; the customer index $i$ there is $k$ here.

In [ ]:
toy_location = {
    "open_cost": np.array([10, 10]),
    "ship_cost": np.array([[0, 100], [100, 0]]),
}

rng = np.random.default_rng(2026)
facility_xy = rng.uniform(0, 100, (10, 2))
customer_xy = rng.uniform(0, 100, (40, 2))
open_cost = rng.integers(150, 300, 10)
distance = np.linalg.norm(facility_xy[:, None, :] - customer_xy[None, :, :], axis=2)
ship_cost = np.rint(distance)
large_location = {"open_cost": open_cost, "ship_cost": ship_cost}
print(f"Larger instance: {len(open_cost)} facilities, {ship_cost.shape[1]} customers")


def location_scaffold(data):
    F, C = data["open_cost"], data["ship_cost"]
    model = pyo.ConcreteModel(name="Uncapacitated facility location")
    model.facilities = pyo.RangeSet(0, len(F) - 1)
    model.customers = pyo.RangeSet(0, C.shape[1] - 1)
    model.open = pyo.Var(model.facilities, domain=pyo.Binary)
    model.allocation = pyo.Var(model.facilities, model.customers, bounds=(0, 1))
    model.objective = pyo.Objective(
        expr=sum(F[j] * model.open[j] for j in model.facilities)
        + sum(C[j, k] * model.allocation[j, k]
              for j in model.facilities for k in model.customers)
    )
    model.serve = pyo.Constraint(
        model.customers,
        rule=lambda m, k: sum(m.allocation[j, k] for j in m.facilities) == 1,
    )
    return model

In [ ]:
# Complete build_location(data, strong=False) from location_scaffold(data).
# Add either the aggregated row or the customer-specific rows.
# Use fresh models in compare_formulation for both instances and both variants.
# Explain the inclusions, fractional counterexample and role of continuous allocations.

What did your checks establish, which errors could they detect, and what remains
unchecked? Distinguish a feasible plan, a proven optimum and an observed change
in solver effort.